In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [6]:
len(words)

32033

In [6]:
#build vocabulary of characters and int mappings
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [9]:
#build dataset
block_size = 3 #context length
X, Y = [], []
for w in words[:5]:
    print(w)
    context = [0]*block_size
    for ch in w+'.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '-->', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... --> e
..e --> m
.em --> m
emm --> a
mma --> .
olivia
... --> o
..o --> l
.ol --> i
oli --> v
liv --> i
ivi --> a
via --> .
ava
... --> a
..a --> v
.av --> a
ava --> .
isabella
... --> i
..i --> s
.is --> a
isa --> b
sab --> e
abe --> l
bel --> l
ell --> a
lla --> .
sophia
... --> s
..s --> o
.so --> p
sop --> h
oph --> i
phi --> a
hia --> .


In [10]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 19],
        [ 9, 19,  1],
        [19,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 12],
        [ 5, 12, 12],
        [12, 12,  1],
        [ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19, 15],
        [19, 15, 16],
        [15, 16,  8],
        [16,  8,  9],
        [ 8,  9,  1]])

In [13]:
# first come up with embedding C X-->N-vector
#27 characters into 2d space
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
C

tensor([[ 1.5674, -0.2373],
        [-0.0274, -1.1008],
        [ 0.2859, -0.0296],
        [-1.5471,  0.6049],
        [ 0.0791,  0.9046],
        [-0.4713,  0.7868],
        [-0.3284, -0.4330],
        [ 1.3729,  2.9334],
        [ 1.5618, -1.6261],
        [ 0.6772, -0.8404],
        [ 0.9849, -0.1484],
        [-1.4795,  0.4483],
        [-0.0707,  2.4968],
        [ 2.4448, -0.6701],
        [-1.2199,  0.3031],
        [-1.0725,  0.7276],
        [ 0.0511,  1.3095],
        [-0.8022, -0.8504],
        [-1.8068,  1.2523],
        [ 0.1476, -1.0006],
        [-0.5030, -1.0660],
        [ 0.8480,  2.0275],
        [-0.1158, -1.2078],
        [-1.0406, -1.5367],
        [-0.5132,  0.2961],
        [-1.4904, -0.2838],
        [ 0.2569,  0.2130]])

In [15]:
#basic way to "embed" a value '5': just index into matrix
C[5]

tensor([-0.4713,  0.7868])

In [20]:
#or use one-hot to encode into binary sequence
example = F.one_hot(torch.tensor(5), num_classes=27).float()
#multiply one hot by C to be equivalent of indexing into it
example @ C

tensor([-0.4713,  0.7868])

In [21]:
#index with a list
C[[5,6,7]]

tensor([[-0.4713,  0.7868],
        [-0.3284, -0.4330],
        [ 1.3729,  2.9334]])

In [24]:
#index with a tensor of integers
C[torch.tensor([5,6,7,7,7])]

tensor([[-0.4713,  0.7868],
        [-0.3284, -0.4330],
        [ 1.3729,  2.9334],
        [ 1.3729,  2.9334],
        [ 1.3729,  2.9334]])

In [27]:
#index with a multidimensional tensor of integers
C[X].shape #for each of 32x3 integers, retreived 2D vector:

torch.Size([32, 3, 2])

In [29]:
print(C[X][13,2])
print(C[1])

tensor([-0.0274, -1.1008])
tensor([-0.0274, -1.1008])


In [33]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [34]:
#construct hidden layer
W1 = torch.randn((6, 100)) #Layer: inputs: 3chars*2Dim embedding, 100 neurons
b1 = torch.randn(100)


In [36]:
#roughly want emb @ W1 + B1
#but this is 32x3x2  @ 6x100 --> Won't work! 
#need to concantenate the inputs (technically many ways to do this)
torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1)


tensor([[ 1.5674, -0.2373,  1.5674, -0.2373,  1.5674, -0.2373],
        [ 1.5674, -0.2373,  1.5674, -0.2373, -0.4713,  0.7868],
        [ 1.5674, -0.2373, -0.4713,  0.7868,  2.4448, -0.6701],
        [-0.4713,  0.7868,  2.4448, -0.6701,  2.4448, -0.6701],
        [ 2.4448, -0.6701,  2.4448, -0.6701, -0.0274, -1.1008],
        [ 1.5674, -0.2373,  1.5674, -0.2373,  1.5674, -0.2373],
        [ 1.5674, -0.2373,  1.5674, -0.2373, -1.0725,  0.7276],
        [ 1.5674, -0.2373, -1.0725,  0.7276, -0.0707,  2.4968],
        [-1.0725,  0.7276, -0.0707,  2.4968,  0.6772, -0.8404],
        [-0.0707,  2.4968,  0.6772, -0.8404, -0.1158, -1.2078],
        [ 0.6772, -0.8404, -0.1158, -1.2078,  0.6772, -0.8404],
        [-0.1158, -1.2078,  0.6772, -0.8404, -0.0274, -1.1008],
        [ 1.5674, -0.2373,  1.5674, -0.2373,  1.5674, -0.2373],
        [ 1.5674, -0.2373,  1.5674, -0.2373, -0.0274, -1.1008],
        [ 1.5674, -0.2373, -0.0274, -1.1008, -0.1158, -1.2078],
        [-0.0274, -1.1008, -0.1158, -1.2

In [39]:
#another way to concatenate
torch.cat(torch.unbind(emb, 1), 1)

tensor([[ 1.5674, -0.2373,  1.5674, -0.2373,  1.5674, -0.2373],
        [ 1.5674, -0.2373,  1.5674, -0.2373, -0.4713,  0.7868],
        [ 1.5674, -0.2373, -0.4713,  0.7868,  2.4448, -0.6701],
        [-0.4713,  0.7868,  2.4448, -0.6701,  2.4448, -0.6701],
        [ 2.4448, -0.6701,  2.4448, -0.6701, -0.0274, -1.1008],
        [ 1.5674, -0.2373,  1.5674, -0.2373,  1.5674, -0.2373],
        [ 1.5674, -0.2373,  1.5674, -0.2373, -1.0725,  0.7276],
        [ 1.5674, -0.2373, -1.0725,  0.7276, -0.0707,  2.4968],
        [-1.0725,  0.7276, -0.0707,  2.4968,  0.6772, -0.8404],
        [-0.0707,  2.4968,  0.6772, -0.8404, -0.1158, -1.2078],
        [ 0.6772, -0.8404, -0.1158, -1.2078,  0.6772, -0.8404],
        [-0.1158, -1.2078,  0.6772, -0.8404, -0.0274, -1.1008],
        [ 1.5674, -0.2373,  1.5674, -0.2373,  1.5674, -0.2373],
        [ 1.5674, -0.2373,  1.5674, -0.2373, -0.0274, -1.1008],
        [ 1.5674, -0.2373, -0.0274, -1.1008, -0.1158, -1.2078],
        [-0.0274, -1.1008, -0.1158, -1.2

In [41]:
#An even better way to concantenate:
a = torch.arange(18)
a

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])

In [43]:
a.view(9,2)

tensor([[ 0,  1],
        [ 2,  3],
        [ 4,  5],
        [ 6,  7],
        [ 8,  9],
        [10, 11],
        [12, 13],
        [14, 15],
        [16, 17]])

In [45]:
emb.shape

torch.Size([32, 3, 2])

In [47]:
emb.view(32,6) == torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1)

tensor([[True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, T

In [61]:
#to make our neuron layer:
#emb.view(32, 6) @ W1 + B1
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) #-1 tells pytorch to figure it out
#be careful with broadcasting between matrix and B1
#32, 100
#    100
(emb.view(-1, 6) @ W1).shape[1] == b1.shape[0]

True

In [62]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

logits = h @ W2 + b2

In [64]:
counts = logits.exp()
prob = counts/counts.sum(1, keepdims=True)
prob.shape

torch.Size([32, 27])

In [67]:
#Actual error (for loss):
loss = -prob[torch.arange(32), Y].log().mean() #negative log likelihood loss
loss

tensor(18.3273)

In [7]:
#REWRITE
#build dataset
block_size = 3 #context length
X, Y = [], []
for w in words:
    context = [0]*block_size
    for ch in w+'.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]
X = torch.tensor(X)
Y = torch.tensor(Y)

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100)) #Layer: inputs: 3chars*2Dim embedding, 100 neurons
b1 = torch.randn(100)
W2 = torch.randn((100, 27))
b2 = torch.randn(27)
parameters = [C, W1, b1, W2, b2]

for p in parameters:
    p.requires_grad=True

for _ in range (50):
    #forward pass
    emb = C[X]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    #manually implement classification math:
    #counts = logits.exp()
    #prob = counts/counts.sum(1, keepdims=True)
    #loss = -prob[torch.arange(32), Y].log().mean()
    
    #use pytorch cross_entropy
    loss = F.cross_entropy(logits, Y)
    print(loss.item())
    
    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    #update
    for p in parameters:
        p.data += -0.1*p.grad

print("Final Loss: ", loss.item())

20.841930389404297
19.40227508544922
18.432205200195312
17.647798538208008
16.926067352294922
16.247873306274414
15.601541519165039
14.982945442199707
14.392614364624023
13.836115837097168
13.321077346801758
12.849050521850586
12.411602020263672
11.999811172485352
11.610344886779785
11.241704940795898
10.892389297485352
10.561270713806152
10.247761726379395
9.951569557189941
9.672453880310059
9.410133361816406
9.164287567138672
8.934518814086914
8.720287322998047
8.520855903625488
8.335256576538086
8.162300109863281
8.000598907470703
7.848696231842041
7.705204010009766
7.568911075592041
7.438805103302002
7.3140740394592285
7.194077014923096
7.07830286026001
6.966364860534668
6.857968807220459
6.7528977394104
6.651001453399658
6.552178859710693
6.456366062164307
6.363532543182373
6.273655414581299
6.186720848083496
6.102705955505371
6.021564483642578
5.943218231201172
5.867551326751709
5.794417381286621
Final Loss:  5.794417381286621


In [8]:
#With batching:
block_size = 3 #context length
X, Y = [], []
for w in words:
    context = [0]*block_size
    for ch in w+'.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]
X = torch.tensor(X)
Y = torch.tensor(Y)

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100)) #Layer: inputs: 3chars*2Dim embedding, 100 neurons
b1 = torch.randn(100)
W2 = torch.randn((100, 27))
b2 = torch.randn(27)
parameters = [C, W1, b1, W2, b2]

for p in parameters:
    p.requires_grad=True

for _ in range (10):
    #minibatch construct
    ix = torch.randint(0, X.shape[0], (32,))
    #forward pass
    emb = C[X[ix]] #again, 32x3x2
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    #manually implement classification math:
    #counts = logits.exp()
    #prob = counts/counts.sum(1, keepdims=True)
    #loss = -prob[torch.arange(32), Y].log().mean()
    
    #use pytorch cross_entropy
    loss = F.cross_entropy(logits, Y[ix])
    print(loss.item())
    
    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    #update
    for p in parameters:
        p.data += -0.1*p.grad

print("Final Loss: ", loss.item())

15.163421630859375
16.17961883544922
16.91411590576172
14.289109230041504
14.808903694152832
12.906107902526855
13.34440803527832
13.517596244812012
11.631810188293457
12.117977142333984
Final Loss:  12.117977142333984


In [ ]:
#loss for all X and Y
    emb = C[X]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    #manually implement classification math:
    #counts = logits.exp()
    #prob = counts/counts.sum(1, keepdims=True)
    #loss = -prob[torch.arange(32), Y].log().mean()
    
    #use pytorch cross_entropy
    loss = F.cross_entropy(logits, Y)
    print(loss.item())